In [33]:
import pandas as pd
import statsmodels.formula.api as smf
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.iolib.summary2 import summary_col as sc
import numpy as np

url = 'https://raw.githubusercontent.com/git-iush/large_family/refs/heads/main/data2.parquet'

In [34]:
df = pd.read_parquet(url)
df.head()

,SERIAL,HHWT,CLUSTER,STRATA,PERNUM,PERWT,NCHILD,SEX,AGE,MARST,...,BLACK,AIAN,API,OTHER,MULTR,HISP,EXPANSION,WELFARE,PRSNINC,YRSEDUC
0,4,4.0,2024000000041,280201,1,4.0,0,2,22,6,...,0,0,1,0,0,0,0,0.0,1000.0,13
1,9,74.0,2024000000091,280301,1,74.0,0,2,18,6,...,1,0,0,0,0,0,0,0.0,5100.0,12
2,12,55.0,2024000000121,160001,1,55.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,0.0,12
3,15,17.0,2024000000151,120201,1,17.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,6000.0,12
4,19,71.0,2024000000191,10001,1,71.0,0,2,23,6,...,0,0,0,0,0,0,0,0.0,0.0,13


In [35]:
df.columns

Index(['SERIAL', 'HHWT', 'CLUSTER', 'STRATA', 'PERNUM', 'PERWT', 'NCHILD',
       'SEX', 'AGE', 'MARST', 'HCOVANY', 'HINSCAID', 'EDUCD', 'INCTOT',
       'FTOTINC', 'INCWELFR', 'CONNETICUT', 'MAINE', 'MASSACHUSETTS',
       'NEWHAMPSHIRE', 'RHODEISLAND', 'VERMONT', 'DELAWARE', 'NEWJERSEY',
       'NEWYORK', 'PENNSYLVANIA', 'ILLINOIS', 'INDIANA', 'MICHIGAN', 'OHIO',
       'WISCONSIN', 'IOWA', 'KANSAS', 'MINNESOTA', 'MISSOURI', 'NEBRASKA',
       'NORTHDAKOTA', 'SOUTHDAKOTA', 'VIRGINIA', 'ALABAMA', 'ARKANSAS',
       'GEORGIA', 'LOUISIANA', 'MISSISSIPPI', 'NORTHCAROLINA', 'SOUTHCAROLINA',
       'TEXAS', 'KENTUCKY', 'MARYLAND', 'OKLAHOMA', 'TENNESSEE',
       'WESTVIRGINIA', 'ARIZONA', 'COLORADO', 'IDAHO', 'MONTANA', 'NEVADA',
       'NEWMEXICO', 'UTAH', 'WYOMING', 'CALIFORNIA', 'OREGON', 'WASHINGTON',
       'ALASKA', 'HAWAII', 'DC', 'BLACK', 'AIAN', 'API', 'OTHER', 'MULTR',
       'HISP', 'EXPANSION', 'WELFARE', 'PRSNINC', 'YRSEDUC'],
      dtype='str')

In [36]:
df['MEDICAID'] = df['HINSCAID'] - 1
df['MEDFARE'] = df['MEDICAID'] * df['WELFARE']

df['MEDICAGE'] = df['MEDICAID'] * df['AGE']
df['WELFAGE'] = df['WELFARE'] * df['AGE']
df['MEDFARAGE'] = df['MEDFARE'] * df['AGE']

df['MEDICAGE2'] = df['MEDICAGE'] ** 2
df['WELFAGE2'] = df['WELFAGE'] ** 2 
df['MEDFARAGE2'] = df['MEDFARAGE'] ** 2

df['FAMINC'] = df['FTOTINC'] - df['INCTOT']
df['PRSNINC2'] = df['PRSNINC'] ** 2
df['FAMINC2'] = df['FAMINC'] ** 2

df['AGE2'] = df['AGE'] ** 2

df['HIGHSCHOOL'] = [1 if i == 63 or i==64 else 0 for i in df['EDUCD']]
df['BACHELORS'] = [1 if i == 101 else 0 for i in df['EDUCD']]
df['ADVANCED'] = [1 if i>=114 else 0 for i in df['EDUCD']]

df['LARGEFAM'] = [1 if i >= 3 else 0 for i in df['NCHILD']]

df['MARD'] = [1 if i==1 or i==2 or i==3 else 0 for i in df['MARST']]

In [37]:
fd = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + CONNETICUT + MAINE + MASSACHUSETTS + NEWHAMPSHIRE + RHODEISLAND + VERMONT + DELAWARE + NEWJERSEY + NEWYORK + PENNSYLVANIA + ILLINOIS + INDIANA + MICHIGAN + OHIO + WISCONSIN + IOWA + KANSAS + MINNESOTA + MISSOURI + NEBRASKA + NORTHDAKOTA + SOUTHDAKOTA + VIRGINIA + ALABAMA + ARKANSAS + GEORGIA + LOUISIANA + MISSISSIPPI + NORTHCAROLINA + SOUTHCAROLINA + TEXAS + KENTUCKY + MARYLAND + OKLAHOMA + TENNESSEE + WESTVIRGINIA + ARIZONA + COLORADO + IDAHO + MONTANA + NEVADA + NEWMEXICO + UTAH + WYOMING + CALIFORNIA + OREGON + WASHINGTON + ALASKA + HAWAII + DC + BLACK + AIAN + API + OTHER + MULTR + HISP + HIGHSCHOOL + BACHELORS + ADVANCED'
fdres = smf.ols(fd, data=df).fit()

sd = 'NCHILD ~ MEDICAGE + MEDICAGE2 + WELFAGE + WELFAGE2 + MEDFARAGE + MEDFARAGE2 + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + CONNETICUT + MAINE + MASSACHUSETTS + NEWHAMPSHIRE + RHODEISLAND + VERMONT + DELAWARE + NEWJERSEY + NEWYORK + PENNSYLVANIA + ILLINOIS + INDIANA + MICHIGAN + OHIO + WISCONSIN + IOWA + KANSAS + MINNESOTA + MISSOURI + NEBRASKA + NORTHDAKOTA + SOUTHDAKOTA + VIRGINIA + ALABAMA + ARKANSAS + GEORGIA + LOUISIANA + MISSISSIPPI + NORTHCAROLINA + SOUTHCAROLINA + TEXAS + KENTUCKY + MARYLAND + OKLAHOMA + TENNESSEE + WESTVIRGINIA + ARIZONA + COLORADO + IDAHO + MONTANA + NEVADA + NEWMEXICO + UTAH + WYOMING + CALIFORNIA + OREGON + WASHINGTON + ALASKA + HAWAII + DC + BLACK + AIAN + API + OTHER + MULTR + HISP + HIGHSCHOOL + BACHELORS + ADVANCED'
sdres = smf.ols(sd, data=df).fit()

did = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + MEDICAGE + MEDICAGE2 + WELFAGE + WELFAGE2 + MEDFARAGE + MEDFARAGE2 + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + CONNETICUT + MAINE + MASSACHUSETTS + NEWHAMPSHIRE + RHODEISLAND + VERMONT + DELAWARE + NEWJERSEY + NEWYORK + PENNSYLVANIA + ILLINOIS + INDIANA + MICHIGAN + OHIO + WISCONSIN + IOWA + KANSAS + MINNESOTA + MISSOURI + NEBRASKA + NORTHDAKOTA + SOUTHDAKOTA + VIRGINIA + ALABAMA + ARKANSAS + GEORGIA + LOUISIANA + MISSISSIPPI + NORTHCAROLINA + SOUTHCAROLINA + TEXAS + KENTUCKY + MARYLAND + OKLAHOMA + TENNESSEE + WESTVIRGINIA + ARIZONA + COLORADO + IDAHO + MONTANA + NEVADA + NEWMEXICO + UTAH + WYOMING + CALIFORNIA + OREGON + WASHINGTON + ALASKA + HAWAII + DC + BLACK + AIAN + API + OTHER + MULTR + HISP + HIGHSCHOOL + BACHELORS + ADVANCED'
didres = smf.ols(did, data=df).fit()




In [53]:
log = 'LARGEFAM ~ MEDICAID + WELFARE + MEDFARE + MEDICAGE + MEDICAGE2 + WELFAGE + WELFAGE2 + MEDFARAGE + MEDFARAGE2 + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + CONNETICUT + MAINE + MASSACHUSETTS + NEWHAMPSHIRE + RHODEISLAND + VERMONT + DELAWARE + NEWJERSEY + NEWYORK + PENNSYLVANIA + ILLINOIS + INDIANA + MICHIGAN + OHIO + WISCONSIN + IOWA + KANSAS + MINNESOTA + MISSOURI + NEBRASKA + NORTHDAKOTA + SOUTHDAKOTA + VIRGINIA + ALABAMA + ARKANSAS + GEORGIA + LOUISIANA + MISSISSIPPI + NORTHCAROLINA + SOUTHCAROLINA + TEXAS + KENTUCKY + MARYLAND + OKLAHOMA + TENNESSEE + WESTVIRGINIA + ARIZONA + COLORADO + IDAHO + MONTANA + NEVADA + NEWMEXICO + UTAH + WYOMING + CALIFORNIA + OREGON + WASHINGTON + ALASKA + HAWAII + DC + BLACK + AIAN + API + OTHER + MULTR + HISP + HIGHSCHOOL + BACHELORS + ADVANCED'

logres = smf.logit(log, data=df).fit()

Optimization terminated successfully.
         Current function value: 0.282125
         Iterations 10


In [39]:
print(didres.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.322
Model:                            OLS   Adj. R-squared:                  0.322
Method:                 Least Squares   F-statistic:                     4305.
Date:                Fri, 13 Nov 2026   Prob (F-statistic):               0.00
Time:                        17:21:58   Log-Likelihood:            -9.6338e+05
No. Observations:              678476   AIC:                         1.927e+06
Df Residuals:                  678400   BIC:                         1.928e+06
Df Model:                          75                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -2.6083      0.017   -150.591

In [54]:
print(logres.summary())

                           Logit Regression Results                           
Dep. Variable:               LARGEFAM   No. Observations:               678476
Model:                          Logit   Df Residuals:                   678400
Method:                           MLE   Df Model:                           75
Date:                Fri, 13 Nov 2026   Pseudo R-squ.:                  0.1930
Time:                        17:23:35   Log-Likelihood:            -1.9141e+05
converged:                       True   LL-Null:                   -2.3719e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept       -21.3803      0.162   -131.969      0.000     -21.698     -21.063
MEDICAID          3.2049      0.267     11.988      0.000       2.681       3.729
WELFARE           1.1388      2.878     

In [44]:
df['MARD'].value_counts()

MARD
0    400453
1    320215
Name: count, dtype: int64

In [67]:
df['yhat_log'] = pd.Series(logres.predict(df), index=logres.model.data.row_labels)
rss_logit = df['yhat_log'].sum() ** 2

In [66]:
df['yhat_did'] = pd.Series(didres.predict(df), index=didres.model.data.row_labels)
rss_did = df['yhat_did'].sum() ** 2

In [65]:
df['yhat_sd'] = pd.Series(sdres.predict(df), index=sdres.model.data.row_labels)
rss_sd = df['yhat_sd'].sum() ** 2

In [64]:
df['yhat_fd'] = pd.Series(fdres.predict(df), index=fdres.model.data.row_labels)
rss_fd = df['yhat_fd'].sum() ** 2

In [69]:
print('Is FD more robust than SD?:',rss_fd<rss_sd)
print('Is SD more robust than DiD?:',rss_sd<rss_did)
print('Is DiD more robust than Logit?:',rss_did<rss_logit)

Is FD more robust than SD?: False
Is SD more robust than DiD?: False
Is DiD more robust than Logit?: False


In [59]:
c = df.columns
c

Index(['SERIAL', 'HHWT', 'CLUSTER', 'STRATA', 'PERNUM', 'PERWT', 'NCHILD',
       'SEX', 'AGE', 'MARST', 'HCOVANY', 'HINSCAID', 'EDUCD', 'INCTOT',
       'FTOTINC', 'INCWELFR', 'CONNETICUT', 'MAINE', 'MASSACHUSETTS',
       'NEWHAMPSHIRE', 'RHODEISLAND', 'VERMONT', 'DELAWARE', 'NEWJERSEY',
       'NEWYORK', 'PENNSYLVANIA', 'ILLINOIS', 'INDIANA', 'MICHIGAN', 'OHIO',
       'WISCONSIN', 'IOWA', 'KANSAS', 'MINNESOTA', 'MISSOURI', 'NEBRASKA',
       'NORTHDAKOTA', 'SOUTHDAKOTA', 'VIRGINIA', 'ALABAMA', 'ARKANSAS',
       'GEORGIA', 'LOUISIANA', 'MISSISSIPPI', 'NORTHCAROLINA', 'SOUTHCAROLINA',
       'TEXAS', 'KENTUCKY', 'MARYLAND', 'OKLAHOMA', 'TENNESSEE',
       'WESTVIRGINIA', 'ARIZONA', 'COLORADO', 'IDAHO', 'MONTANA', 'NEVADA',
       'NEWMEXICO', 'UTAH', 'WYOMING', 'CALIFORNIA', 'OREGON', 'WASHINGTON',
       'ALASKA', 'HAWAII', 'DC', 'BLACK', 'AIAN', 'API', 'OTHER', 'MULTR',
       'HISP', 'EXPANSION', 'WELFARE', 'PRSNINC', 'YRSEDUC', 'MEDICAID',
       'MEDFARE', 'MEDICAGE', 'WELFAGE'

In [132]:
mf = pd.DataFrame()
mf['Column'] = c
mf['Mean'] = [df[i].mean() for i in c]

In [133]:
mf.index = c
mf = mf.drop(columns='Column')
mf.head()

,Mean
SERIAL,7.634913e+05
HHWT,1.092335e+02
CLUSTER,2.024008e+12
STRATA,5.048644e+05
PERNUM,1.967820e+00


In [134]:
print(df['AGE'].mean() == mf.loc['AGE'])
print(df['MEDICAID'].mean() == mf.loc['MEDICAID'])

Mean    True
Name: AGE, dtype: bool
Mean    True
Name: MEDICAID, dtype: bool
